# Fine-tune SVTR26-Tiny Đa năng: Kết hợp cả Ô tô + Xe máy Việt Nam

- **Trọng số khởi tạo:** Huấn luyện tiếp từ chính mô hình Fine-tune Xe máy của bạn (`/content/drive/MyDrive/LITEALPR_FINETUNE_VN/best.pth` — chống quên kiến thức xe máy, bổ sung kiến thức ô tô).
- **Tập dữ liệu `tap_finetune_hon_hop.zip` (tỷ lệ cân bằng ~50/50):**
  - `Train`: **2.416 ảnh** (`1.275` Xe máy + `1.141` Ô tô từ 4 ngày `02..05/08`, đã xáo trộn đều).
  - `Val`: **271 ảnh** (`124` Xe máy + `147` Ô tô) — giúp tự động lưu `best.pth` tốt nhất trên cả 2 loại xe.
  - `Test độc lập`: **672 ảnh** (`128` Xe máy độc lập + `544` Ô tô ngày `01/08` hoàn toàn không trùng xe).
- **Đầu ra:** Gộp gọn vào **1 file duy nhất `svtr26_hon_hop.onnx`** lưu thẳng vào Google Drive.

In [ ]:
# Bước 1: Kết nối Google Drive và kiểm tra GPU
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi

In [ ]:
# Bước 2: Tải mã nguồn LiteALPR, cài thư viện và vá tương thích NumPy 2.0+
%cd /content
!rm -rf /content/LiteALPR
!git clone https://github.com/vn-anhnth/LiteALPR.git
%cd /content/LiteALPR
!pip install -q ultralytics imgaug pyclipper onnxscript onnxslim lmdb rapidfuzz onnx onnxruntime-gpu huggingface_hub pyyaml tqdm pillow opencv-python
# Vá tương thích imgaug 0.4.0 trên NumPy 2.0+ bằng find_spec (không import imgaug trước khi vá!)
import importlib.util, pathlib
p_imgaug = pathlib.Path(importlib.util.find_spec('imgaug').submodule_search_locations[0]) / 'imgaug.py'
noi_dung = p_imgaug.read_text()
ban_va = 'if not hasattr(np, "sctypes"): np.sctypes = {"int":[np.int8,np.int16,np.int32,np.int64],"uint":[np.uint8,np.uint16,np.uint32,np.uint64],"float":[np.float16,np.float32,np.float64],"complex":[np.complex64,np.complex128],"others":[bool,object,bytes,str,np.void]}\n'
if 'hasattr(np, "sctypes")' not in noi_dung:
    p_imgaug.write_text(noi_dung.replace('import numpy as np\n', 'import numpy as np\n' + ban_va, 1))
print('✅ Đã vá imgaug thành công!')

In [ ]:
# Bước 3: Giải nén tap_finetune_hon_hop.zip (hãy upload file tap_finetune_hon_hop.zip lên MyDrive trước)
DUONG_DAN_ZIP = '/content/drive/MyDrive/tap_finetune_hon_hop.zip'

!rm -rf /content/du_lieu_hon_hop /content/lmdb_hon_hop
!mkdir -p /content/du_lieu_hon_hop
!unzip -q -o "{DUONG_DAN_ZIP}" -d /content/du_lieu_hon_hop
!wc -l /content/du_lieu_hon_hop/*_labels.txt

In [ ]:
# Bước 4: Đóng gói sang định dạng LMDB (cho cả Train, Val và 3 tập Test)
%cd /content/LiteALPR
!python tools/create_lmdb_dataset.py \
  --data_dir /content/du_lieu_hon_hop \
  --label_files train_labels.txt val_labels.txt test_labels.txt test_xemay_labels.txt test_oto_labels.txt \
  --output_dir /content/lmdb_hon_hop
!ls -lh /content/lmdb_hon_hop

In [ ]:
# Bước 5: Nạp trọng số SVTR26 Fine-tuned Xe máy (best.pth vòng 1) & đo mốc TRƯỚC khi train
from pathlib import Path
import shutil
from huggingface_hub import hf_hub_download

PTH_XE_MAY = Path('/content/drive/MyDrive/LITEALPR_FINETUNE_VN/best.pth')
TRONG_SO_KHOI_TAO = '/content/pretrained_xemay.pth'

if PTH_XE_MAY.exists():
    shutil.copy(PTH_XE_MAY, TRONG_SO_KHOI_TAO)
    print('✅ Đã nạp trọng số SVTR26 Fine-tuned Xe máy:', PTH_XE_MAY)
else:
    duong_dan_goc = hf_hub_download(repo_id='anhone3/LiteALPR', filename='svtr26_tiny/best.pth')
    shutil.copy(duong_dan_goc, TRONG_SO_KHOI_TAO)
    print('⚠️ Không thấy LITEALPR_FINETUNE_VN/best.pth -> Dùng svtr26_tiny/best.pth gốc')

%cd /content/LiteALPR
print('=== Mốc TRƯỚC khi train trên Test Xe máy (128 ảnh) ===')
!python tools/eval_rec.py -c configs/rec/svtr26/svtr26_tiny.yml -o Global.pretrained_model={TRONG_SO_KHOI_TAO} Eval.loader.batch_size_per_card=64 Eval.sampler.first_bs=64 Eval.dataset.data_dir_list="['/content/lmdb_hon_hop/test_xemay']"
print('=== Mốc TRƯỚC khi train trên Test Ô tô ngày 01/08 (544 ảnh) ===')
!python tools/eval_rec.py -c configs/rec/svtr26/svtr26_tiny.yml -o Global.pretrained_model={TRONG_SO_KHOI_TAO} Eval.loader.batch_size_per_card=64 Eval.sampler.first_bs=64 Eval.dataset.data_dir_list="['/content/lmdb_hon_hop/test_oto']"

In [ ]:
# Bước 6: Huấn luyện kết hợp Ô tô + Xe máy (22 epochs, lr=0.00012, batch_size=64)
THU_MUC_LUU_DRIVE = '/content/drive/MyDrive/LITEALPR_FINETUNE_HON_HOP'
!mkdir -p "{THU_MUC_LUU_DRIVE}"

%cd /content/LiteALPR
!python tools/train_rec.py \
  -c configs/rec/svtr26/svtr26_tiny.yml \
  -o Global.pretrained_model=/content/pretrained_xemay.pth \
     Global.output_dir="{THU_MUC_LUU_DRIVE}" \
     Global.epoch_num=22 \
     Global.eval_epoch_step="[0, 1]" \
     LRScheduler.warmup_epoch=2 \
     Optimizer.lr=0.00012 \
     Train.loader.batch_size_per_card=64 \
     Train.sampler.first_bs=64 \
     Eval.loader.batch_size_per_card=64 \
     Eval.sampler.first_bs=64 \
     Train.dataset.data_dir_list="['/content/lmdb_hon_hop/train']" \
     Eval.dataset.data_dir_list="['/content/lmdb_hon_hop/val']"

In [ ]:
# Bước 7: Đánh giá SAU Fine-tune trên từng tập Test riêng biệt & Xuất 1 file duy nhất svtr26_hon_hop.onnx
%cd /content/LiteALPR
print('=== KẾT QUẢ SAU FINE-TUNE: Test Xe máy độc lập (128 ảnh) ===')
!python tools/eval_rec.py -c configs/rec/svtr26/svtr26_tiny.yml -o Global.pretrained_model="{THU_MUC_LUU_DRIVE}/best.pth" Eval.loader.batch_size_per_card=64 Eval.sampler.first_bs=64 Eval.dataset.data_dir_list="['/content/lmdb_hon_hop/test_xemay']"

print('=== KẾT QUẢ SAU FINE-TUNE: Test Ô tô độc lập ngày 01/08 (544 ảnh) ===')
!python tools/eval_rec.py -c configs/rec/svtr26/svtr26_tiny.yml -o Global.pretrained_model="{THU_MUC_LUU_DRIVE}/best.pth" Eval.loader.batch_size_per_card=64 Eval.sampler.first_bs=64 Eval.dataset.data_dir_list="['/content/lmdb_hon_hop/test_oto']"

print('=== KẾT QUẢ TỔNG HỢP CẢ HAI (672 ảnh) ===')
!python tools/eval_rec.py -c configs/rec/svtr26/svtr26_tiny.yml -o Global.pretrained_model="{THU_MUC_LUU_DRIVE}/best.pth" Eval.loader.batch_size_per_card=64 Eval.sampler.first_bs=64 Eval.dataset.data_dir_list="['/content/lmdb_hon_hop/test']"

!mkdir -p /content/onnx_tmp
!python tools/export_rec.py \
  -m "{THU_MUC_LUU_DRIVE}/best.pth" \
  --save_path /content/onnx_tmp/raw.onnx \
  --dynamic

import onnx
m = onnx.load('/content/onnx_tmp/raw.onnx', load_external_data=True)
duong_dan_onnx_cuoi = f'{THU_MUC_LUU_DRIVE}/svtr26_hon_hop.onnx'
onnx.save_model(m, duong_dan_onnx_cuoi, save_as_external_data=False)
print('✅ ĐÃ XUẤT XONG FILE DUY NHẤT:', duong_dan_onnx_cuoi)
!ls -lh "{duong_dan_onnx_cuoi}"